# T-05: Cache Qwen2.5-VL-3B-Instruct
> Run on **Kaggle GPU (T4 or P100)** — model is ~6 GB

**Steps:** Install -> Download -> Smoke test -> Verify offline -> Zip for team

**Why VL not text-only Qwen3?**  
Your `qwen3-4B-text.ipynb` is text-only. `Qwen2.5-VL` understands images — it can read brand, weight, dimensions directly from product packaging photos.

In [ ]:
# STEP 1: Install required packages
!pip install -q 'transformers>=4.45.0' accelerate qwen-vl-utils huggingface_hub bitsandbytes
print('All packages installed')

In [ ]:
# STEP 2: GPU verification
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name      :', torch.cuda.get_device_name(0))
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print('VRAM          : {:.1f} GB'.format(vram))
print('PyTorch       :', torch.__version__)
# Use 4-bit quantization only if less than 8 GB VRAM
USE_4BIT = vram < 8
print('4-bit mode    :', USE_4BIT)

In [ ]:
# STEP 3: Download Qwen2.5-VL-3B-Instruct
# Size: ~6 GB | Expected time: 15-30 min on Kaggle
import os
from huggingface_hub import snapshot_download

MODEL_ID  = 'Qwen/Qwen2.5-VL-3B-Instruct'
LOCAL_DIR = '/kaggle/working/pretrained_models/vlm/qwen25-vl-3b'
os.makedirs(LOCAL_DIR, exist_ok=True)

print('Downloading:', MODEL_ID)
print('Destination:', LOCAL_DIR)

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=LOCAL_DIR,
    ignore_patterns=['*.md', '*.txt'],
)

files = os.listdir(LOCAL_DIR)
total = sum(
    os.path.getsize(os.path.join(LOCAL_DIR, f))
    for f in files if os.path.isfile(os.path.join(LOCAL_DIR, f))
)
print('Files downloaded:', len(files))
print('Total size: {:.2f} GB'.format(total / 1e9))
print('Download complete!')

In [ ]:
# STEP 4: Load model from local cache
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        LOCAL_DIR,
        quantization_config=bnb,
        device_map='auto',
    )
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        LOCAL_DIR,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

processor = AutoProcessor.from_pretrained(LOCAL_DIR)
dtype  = str(next(model.parameters()).dtype)
device = str(next(model.parameters()).device)
print('Model loaded successfully!')
print('dtype :', dtype)
print('device:', device)

In [ ]:
# STEP 5: Smoke test — Amazon-style product attribute extraction
# This is EXACTLY how you will call it on competition day
from qwen_vl_utils import process_vision_info

PROMPT = (
    'Look at this Amazon product image carefully. '
    'Extract: (1) brand name, (2) weight or volume with units, (3) product category. '
    'Return as JSON with keys: brand, weight, category.'
)

def extract_attributes(image_url, model, processor):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image_url},
            {'type': 'text',  'text': PROMPT},
        ]
    }]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
        )

    trimmed = out_ids[0][inputs.input_ids.shape[1]:]
    return processor.decode(trimmed, skip_special_tokens=True)


# Sample Amazon product URL
SAMPLE_URL = 'https://m.media-amazon.com/images/I/71XdyguNoEL._SL1500_.jpg'
result = extract_attributes(SAMPLE_URL, model, processor)

print('=== Model Output ===')
print(result)
print('====================')
print('Smoke test PASSED')

In [ ]:
# STEP 6: Verify OFFLINE loading (simulates competition day with internet OFF)
# local_files_only=True ensures ZERO network calls
import gc
del model, processor
gc.collect()
torch.cuda.empty_cache()
print('Model cleared from memory')

model_offline = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    LOCAL_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=True,   # <-- NO internet
)
proc_offline = AutoProcessor.from_pretrained(LOCAL_DIR, local_files_only=True)

param_B = sum(p.numel() for p in model_offline.parameters()) / 1e9
print('Offline load VERIFIED — local_files_only=True works')
print('Model parameters: {:.1f}B'.format(param_B))

In [ ]:
# STEP 7: Zip model for Kaggle Dataset upload (share with entire team)
import zipfile

ZIP_PATH = '/kaggle/working/qwen25-vl-3b.zip'
print('Zipping model files...')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for root, dirs, files_list in os.walk(LOCAL_DIR):
        for fname in files_list:
            fp  = os.path.join(root, fname)
            arc = os.path.relpath(fp, '/kaggle/working/pretrained_models/vlm/')
            zf.write(fp, arc)

zip_gb = os.path.getsize(ZIP_PATH) / 1e9
print('Zip created: {:.2f} GB'.format(zip_gb))
print()
print('=== HOW TO SHARE WITH TEAM ===')
print('1. Download zip from Kaggle output (right sidebar -> Output tab)')
print('2. Go to kaggle.com/datasets -> New Dataset')
print('3. Upload the zip file')
print('4. Name: amazon-ml-vlm-cache | Visibility: Private')
print('5. Share dataset with all 4 team members via Collaborators tab')
print('6. Competition day: Add dataset -> files at /kaggle/input/amazon-ml-vlm-cache/')
print()
print('=' * 55)
print('T-05 COMPLETE: Qwen2.5-VL-3B cached and ready!')
print('=' * 55)

## Competition Day — Load Code

```python
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

# From Kaggle Dataset cache
MODEL_PATH = '/kaggle/input/amazon-ml-vlm-cache/qwen25-vl-3b'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=True,
)
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
```

## RTX 4060 Local Load (4-bit quantization)

```python
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    './pretrained_models/vlm/qwen25-vl-3b',
    quantization_config=bnb,
    device_map='auto',
    local_files_only=True,
)
```